# 11 · Quantum image generators and a classical comparison

Train two different image QGAN architectures: quantum pixel patches, and
quantum features feeding a classical decoder. Compare a conditional classical
GAN on the same data, minibatch order, noise prior, discriminator and updates.
This short offline lesson uses vertical/horizontal bars so it runs without a
dataset download. For real cat/dog images use the folder/CIFAR-10 commands below.
No pretrained model is included, and nothing submits work to a QPU.

From the **current source checkout**, in your terminal (Windows supported):

```bash
python -m pip install torch --index-url https://download.pytorch.org/whl/cpu
python -m pip install -e ".[tutorial,image-gan]"
python -m ipykernel install --user --name nexqus --display-name "Python (NexQus)"
```

Restart the kernel and select **Python (NexQus)**. These new optional APIs are
not in the published 1.2.0 wheel, and are included in version 1.4.0.
The chemistry extra and PySCF are not needed. If PyTorch is absent this optional
lesson explicitly skips; image-GAN CI executes it with PyTorch installed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import nexqus
from nexqus import Circuit, Simulator, Parameter, Noise, NoiseModel, algorithms
assert nexqus.__version__ == '1.4.0', 'Install NexQus SDK 1.4.0 in this kernel environment and restart it'
print('NexQus SDK:', nexqus.__version__)


In [ ]:
import importlib.util
import sys
print('Python:', sys.executable)
print('Imported SDK:', nexqus.__file__)
image_gan_available = importlib.util.find_spec('torch') is not None
if image_gan_available:
    try:
        import torch
        from IPython.display import display
        from nexqus.image_gan import (QuantumPatchGenerator, HybridImageGenerator,
            ClassicalImageGenerator, ImageGAN, synthetic_bars)
    except ImportError as error:
        raise ImportError('Install the current source checkout with .[tutorial,image-gan] in this kernel and restart it') from error
    torch.set_num_threads(1)
    from nexqus.torch_circuit import tensor_device
    image_device = tensor_device(None)  # set NEXQUS_DEVICE before starting the kernel for CUDA
    print('Image execution device:', image_device)
else:
    print('SKIPPED optional image GAN: install .[image-gan] to run this lesson')


## Train all three models

The patch generator maps each circuit's ancilla=0 probabilities into a pixel
patch, then tiles the patches. Each patch is normalized to a brightest pixel of
one. The hybrid generator reads Z expectations from its circuit and sends them
through a trainable classical decoder. Noise/class inputs reach that decoder
through quantum features. Both have trainable classical angle encoders.

Circuit states, batched angles and derivatives use native Torch autograd on
CPU/CUDA. Set NEXQUS_DEVICE or pass device='cuda:0' to ImageGAN. The classical
GAN and hybrid decoder use spatial convolutions in this lesson. The API keeps
MLP defaults for old code; specify architecture='conv'. Parameter
budgets differ; they are printed rather than implied to be matched.


In [ ]:
if image_gan_available:
    dataset = synthetic_bars(samples=16, image_size=4, seed=7)
    training, validation = dataset.split(seed=7)
    classes = (QuantumPatchGenerator, HybridImageGenerator, ClassicalImageGenerator)
    trainers = {}
    for constructor in classes:
        options = {} if constructor == QuantumPatchGenerator else {'architecture': 'conv', 'hidden': 32}
        generator = constructor(image_size=4, latent_dim=2, seed=7, **options)
        trainer = ImageGAN(generator, seed=7, device=image_device,
                           discriminator_architecture='conv', discriminator_hidden=32,
                           training='regularized', generator_learning_rate=.0002,
                           discriminator_learning_rate=.0001, r1_gamma=1., r1_every=16,
                           diff_augment=True, ema_decay=.995)
        before = trainer.sample(8, seed=19)
        trainer.fit(training, epochs=1, batch_size=2, max_steps=2)
        after = trainer.sample(8, seed=19)
        assert trainer.steps == 2 and torch.isfinite(after).all()
        assert not torch.equal(before, after)
        name = generator.config['kind']
        trainers[name] = trainer
        print(name, 'parameters:', sum(p.numel() for p in generator.parameters()),
              'quantum:', sum(q.weights.numel() for q in generator.quantum_layers()))
        fig, axes = plt.subplots(2, 8, figsize=(10, 3))
        for row, images in enumerate((before, after)):
            for i, image in enumerate(images):
                axes[row, i].imshow(image[0].cpu(), cmap='gray', vmin=0, vmax=1)
                axes[row, i].axis('off')
        fig.suptitle(name+' · before / after two updates · labels alternate vertical/horizontal')
        plt.tight_layout(); plt.show()


## Change one quantum parameter and see the pixels change

Hold noise and requested class fixed. The first three panels are actual outputs
at w−δ, w, w+δ. The last is their central finite-difference pixel sensitivity.
This experiment shows a local causal intervention in the model. It does not
explain an entire dataset or establish that the model learned animal semantics.


In [ ]:
if image_gan_available:
    for name in ('patch', 'hybrid'):
        generator = trainers[name].sampling_generator()
        noise = torch.tensor([[.2, -.4]], device=image_device)
        label = torch.tensor([0], device=image_device)
        effect = generator.parameter_effect(noise, label, delta=.25)
        fig, axes = plt.subplots(1, 4, figsize=(10, 3))
        for ax, key in zip(axes[:3], ('minus', 'center', 'plus')):
            ax.imshow(effect[key][0, 0].cpu(), cmap='gray', vmin=0, vmax=1); ax.set_title(key); ax.axis('off')
        sensitivity = effect['sensitivity'][0, 0].cpu().numpy()
        bound = max(float(np.max(np.abs(sensitivity))), 1e-9)
        artist = axes[3].imshow(sensitivity, cmap='RdBu_r', vmin=-bound, vmax=bound)
        axes[3].set_title('Pixel change / radian'); axes[3].axis('off')
        fig.colorbar(artist, ax=axes[3], shrink=.65)
        fig.suptitle(name+' · '+effect['parameter']); plt.tight_layout(); plt.show()
        display(generator.circuit_report(noise, label))


## Checkpoint and continue

Checkpoints contain the model, discriminator, optimizers, training RNG, shuffled
data order and cursor. Resume with the same preprocessed data and batch size.
Fixed-seed sampling uses a separate RNG and does not change future training.


In [ ]:
if image_gan_available:
    import tempfile
    from pathlib import Path
    with tempfile.TemporaryDirectory() as directory:
        checkpoint = Path(directory)/'hybrid.pt'
        trainers['hybrid'].save(checkpoint)
        loaded = ImageGAN.load(checkpoint)
        torch.testing.assert_close(loaded.sample(4), trainers['hybrid'].sample(4), rtol=0, atol=0)
        loaded.fit(training, epochs=1, batch_size=2, max_steps=1)
        assert loaded.steps == 3
    print('Checkpoint round trip and continued training passed')


## Your own cats and dogs

Put photos under `data/pets/cats/` and `data/pets/dogs/`. The loader center-crops,
resizes and scales RGB pixels to [0,1]. Use the terminal example for longer runs:

```bash
python examples/image_gan.py --data data/pets --mode classical --architecture dcgan --training regularized --size 64 --channels 3 --samples 10000 --epochs 100 --batch-size 64 --hidden 64 --latent-dim 64 --learning-rate 0.0002 --data-seed 7 --horizontal-flip --output-dir results/pets-dcgan-classical
```

Or optionally install compatible torchvision from the same CPU index and use
`--cifar10 data/cifar10 --download` instead of `--data data/pets`.
The regularized preset adds separate learning rates, R1, symmetric translations
and EMA sampling. dcgan requires fresh weights. The hybrid CLI preset uses
probability readout and linear angles; --latent-skip adds a classical noise
bypass whose contribution must be assessed separately. The report shows
generated images first and saves real/before/after images, losses, checkpoints,
duplicate-output diagnostics, held-out pixel
diagnostics, parameter sensitivity plots and gate-by-gate circuit reports.
`report.html` ties the images and settings together. See `docs/IMAGE_GANS.md`.

The CLI now defaults to a convolutional decoder and discriminator. Replace
--mode classical with --mode hybrid --qubits 6 --depth 2 --latent-skip for the quantum-feature
comparison, using a separate output directory. These are starting experiments;
recognizable pet generation has not been validated. Periodic fixed-noise
previews and checkpoints help inspect training before it completes.
Use --resume with --retune to change training controls while preserving weights
and Adam moments. See docs/IMAGE_GANS.md for bounded retuning and checkpoint
compatibility. Two updates verify the training path only. GAN losses and mean
pixel errors alone do not measure semantic image quality.

**Try it:** increase the update count, inspect both class labels at fixed noise,
and compare patch/hybrid/classical outputs. Investigate whether changing a
quantum parameter affects one patch or the entire decoded image.
